# 01. 데이터 분리 실험 일지

## Step 0. 이 노트북을 작성하는 이유

이번 과제에서는 모델을 학습시키는 것뿐 아니라, 데이터를 어떻게 준비하고 나누었는지 설명하는 과정이 중요합니다.

그래서 이 노트북에서는 Oxford-IIIT Pet Dataset을 dog/cat 이진 분류 문제로 재구성하고, train/validation/test를 직접 나누는 과정을 기록합니다. 단순히 코드를 실행하는 것이 아니라, 각 단계에서 왜 그렇게 판단했는지 제 말로 설명할 수 있게 정리합니다.

## Step 1. 데이터셋 선택을 정리합니다

### 선택지

- **Kaggle Cat and Dog:** 강아지/고양이 분류에 바로 사용할 수 있고, train/test가 이미 나뉘어 있어서 편합니다.
- **Oxford-IIIT Pet:** 강아지/고양이뿐 아니라 품종 라벨도 포함되어 있고, train/validation/test를 직접 나누어 볼 수 있습니다.

### 선택

저는 **Oxford-IIIT Pet Dataset**을 사용합니다.

### 이유

Kaggle 데이터셋은 바로 사용하기 편하지만, 이미 train/test가 나뉘어 있어서 제가 데이터 분리 과정을 직접 경험하기에는 아쉬움이 있다고 판단했습니다.

반면 Oxford-IIIT Pet Dataset은 품종 라벨을 포함하고 있어 dog/cat 이진 분류로 변환하는 과정부터 직접 해볼 수 있습니다. 또한 train/validation/test를 제가 직접 나누면서 각 데이터셋의 역할을 더 분명하게 이해할 수 있습니다.

추가로, 이번 과제에서는 dog/cat 이진 분류만 진행하지만 나중에 발전시키면 품종까지 포함한 다중 클래스 분류 문제로 확장할 수 있다는 점도 장점이라고 생각했습니다.

## Step 2. train / validation / test의 역할을 정리합니다

### 정리

초보자 관점에서는 세 데이터를 다음과 같이 이해합니다.

- **train set:** 모델이 공부하는 문제집입니다.
- **validation set:** 학습 중 선택을 점검하는 모의고사입니다.
- **test set:** 마지막에 한 번만 보는 최종 시험입니다.

중요한 점은 test set을 학습 중에 자주 보면 안 된다는 것입니다. test set은 모델을 다 만든 뒤 마지막 성능 확인에 사용해야 합니다.

## Step 3. 필요한 라이브러리를 불러옵니다

이번 노트북에서는 데이터셋 로드, 표 정리, 데이터 분리, 그래프 저장을 수행합니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision.datasets import OxfordIIITPet

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(PROJECT_ROOT)

## 4. 데이터셋 로드

Oxford-IIIT Pet Dataset은 37개 품종으로 구성되어 있습니다. 이번 과제에서는 우선 dog/cat 이진 분류를 진행하지만, 나중에 품종 분류 문제로 확장할 가능성도 고려합니다. 그래서 원본 데이터에서 이미지 경로뿐 아니라 품종 이름도 함께 가져옵니다.

처음 실행할 때는 `download=True` 때문에 데이터 다운로드가 진행된다.

In [ ]:
dataset = OxfordIIITPet(
    root=RAW_DIR,
    split="trainval",
    target_types="category",
    download=True,
)

print(f"number of images: {len(dataset)}")
print(f"number of classes: {len(dataset.classes)}")
print(dataset.classes[:5])

## 5. dog/cat 라벨 변환 규칙

Oxford-IIIT Pet의 클래스 이름은 품종 이름입니다. 처음에는 품종 이름의 대소문자로 dog/cat을 구분할 수 있다고 생각했지만, `torchvision`에서 제공하는 `dataset.classes`는 `American Bulldog`, `Beagle`처럼 품종 이름을 보기 좋게 정리해서 모두 대문자로 시작합니다.

예시:

- `dataset.classes`는 37개 품종 이름을 담고 있습니다.
- `dataset._labels`는 각 이미지의 품종 번호를 담고 있습니다.
- `dataset._bin_labels`는 각 이미지의 cat/dog 번호를 담고 있습니다.
- `dataset.bin_classes`는 cat/dog 번호를 실제 이름으로 바꾸는 목록입니다.

따라서 이진 라벨은 품종 이름의 첫 글자가 아니라, 데이터셋에 함께 들어 있는 binary label 정보를 사용해서 만듭니다.

In [ ]:
records = []
for image_path, class_index, binary_index in zip(
    dataset._images,
    dataset._labels,
    dataset._bin_labels,
):
    breed_name = dataset.classes[class_index]
    binary_label = dataset.bin_classes[binary_index].lower()
    records.append(
        {
            "image_path": str(image_path),
            "breed": breed_name,
            "label": binary_label,
        }
    )

df = pd.DataFrame(records)
df.head()

## 6. 원본 라벨 분포 확인

데이터를 나누기 전에 먼저 전체 데이터에서 cat/dog 개수가 어느 정도인지 확인합니다. 만약 한쪽 클래스가 너무 적다면 accuracy만으로 성능을 판단하기 어려워질 수 있습니다.

In [ ]:
overall_distribution = df["label"].value_counts().rename_axis("label").reset_index(name="count")
overall_distribution

## 7. split 비율 후보 결정

**선택지:**

- 80/10/10: train이 많아서 학습에는 유리하지만 validation/test가 작습니다.
- 70/15/15: 학습 데이터도 충분히 남기고, 검증/평가 데이터도 어느 정도 확보합니다.
- 60/20/20: 검증/평가는 넉넉하지만 학습 데이터가 줄어듭니다.

**선택한 방향:** 70/15/15를 기본 split으로 사용하고, 80/10/10과 60/20/20도 비교용으로 함께 만들어 둡니다.

**이유:** 이번 과제는 성능 최고점보다 split의 역할을 설명하고 평가를 해석하는 것이 중요합니다. 그래서 기본 실험에는 70/15/15를 사용하되, 나중에 학습 결과를 비교할 수 있도록 80/10/10과 60/20/20 split도 함께 저장합니다.

## 8. 품종 기준 stratified split 생성

`stratify`를 사용하면 특정 기준의 분포가 train/validation/test에 비슷하게 들어가도록 나눌 수 있습니다.

처음에는 dog/cat 라벨 기준으로 나누는 방법을 생각했지만, 이 데이터셋에는 품종 라벨도 있습니다. 특정 품종이 train이나 test에 한쪽으로 몰리면 나중에 품종별 실패 원인을 분석하기 어려울 수 있습니다.

그래서 이번 split은 `label`이 아니라 `breed` 기준으로 stratified split을 만듭니다. 이렇게 하면 dog/cat 분포뿐 아니라 품종 분포도 더 고르게 유지할 수 있습니다.

In [ ]:
# 비교해 볼 train/validation/test 비율 후보를 준비합니다.
SPLIT_CONFIGS = [
    {"name": "split_80_10_10", "train": 0.80, "validation": 0.10, "test": 0.10},
    {"name": "split_70_15_15", "train": 0.70, "validation": 0.15, "test": 0.15},
    {"name": "split_60_20_20", "train": 0.60, "validation": 0.20, "test": 0.20},
]

# 이후 학습 실험에서 기본으로 사용할 split 비율을 선택합니다.
PRIMARY_SPLIT_NAME = "split_70_15_15"
# dog/cat 라벨뿐 아니라 품종 분포까지 비슷하게 나누기 위해 breed를 기준으로 사용합니다.
STRATIFY_COLUMN = "breed"


def make_stratified_split(source_df, config, stratify_column):
    # config에서 이번 split 비율을 꺼냅니다.
    train_ratio = config["train"]
    validation_ratio = config["validation"]
    test_ratio = config["test"]

    # 먼저 전체 데이터에서 train set을 분리합니다.
    # stratify를 사용하면 품종 분포가 전체 데이터와 비슷하게 유지됩니다.
    train_df, temp_df = train_test_split(
        source_df,
        train_size=train_ratio,
        random_state=RANDOM_SEED,
        stratify=source_df[stratify_column],
    )

    # 남은 temp_df 안에서 validation과 test를 나눌 비율을 계산합니다.
    # 예: 70/15/15에서는 temp_df가 30%이고, 그중 절반을 validation으로 사용합니다.
    validation_fraction = validation_ratio / (validation_ratio + test_ratio)
    # temp_df를 validation set과 test set으로 다시 분리합니다.
    validation_df, test_df = train_test_split(
        temp_df,
        train_size=validation_fraction,
        random_state=RANDOM_SEED,
        stratify=temp_df[stratify_column],
    )

    # 세 데이터셋을 하나의 표로 합치기 전에 split 이름을 기록합니다.
    split_parts = []
    for split_name, split_part in [
        ("train", train_df),
        ("validation", validation_df),
        ("test", test_df),
    ]:
        split_part = split_part.copy()
        split_part["split_config"] = config["name"]
        split_part["split"] = split_name
        split_part["stratify_column"] = stratify_column
        split_parts.append(split_part)

    # train/validation/test를 하나의 DataFrame으로 합칩니다.
    result_df = pd.concat(split_parts, ignore_index=True)
    # 이후 단계에서 필요한 컬럼만 정리해서 반환합니다.
    return result_df[["split_config", "split", "stratify_column", "label", "breed", "image_path"]]


# 준비한 모든 split 비율에 대해 같은 방식으로 데이터셋을 나눕니다.
split_tables = {
    config["name"]: make_stratified_split(df, config, STRATIFY_COLUMN)
    for config in SPLIT_CONFIGS
}

# 모든 split 결과를 합친 표와, 대표 split 하나를 따로 만듭니다.
all_splits_df = pd.concat(split_tables.values(), ignore_index=True)
split_df = split_tables[PRIMARY_SPLIT_NAME]
split_df.head()

## 9. split 결과 저장

이후 학습 노트북에서는 같은 split을 그대로 사용해야 합니다. 그래서 split 결과를 CSV로 저장합니다. 이렇게 하면 매번 데이터가 다르게 나뉘는 문제를 피할 수 있습니다.

여기서는 모든 split 후보를 하나의 파일로 저장하고, 각 비율별 split 파일도 따로 저장합니다.

In [ ]:
all_splits_path = PROCESSED_DIR / "pet_binary_splits_all_configs.csv"
all_splits_df.to_csv(all_splits_path, index=False)

split_paths = {}
for split_name, split_table in split_tables.items():
    split_path = PROCESSED_DIR / f"pet_binary_{split_name}_breed_stratified.csv"
    split_table.to_csv(split_path, index=False)
    split_paths[split_name] = split_path

print(all_splits_path)
for split_name, split_path in split_paths.items():
    print(split_name, "->", split_path)

## 10. split별 dog/cat 및 품종 분포 확인

잘 나뉘었는지 보기 위해 split별 dog/cat 개수를 표로 확인합니다. 또한 이번에는 품종 기준으로 stratified split을 만들었기 때문에, 품종별 분포도 함께 저장합니다.

In [ ]:
label_distribution = (
    all_splits_df.groupby(["split_config", "split", "label"])
    .size()
    .rename("count")
    .reset_index()
)

breed_distribution = (
    all_splits_df.groupby(["split_config", "split", "breed"])
    .size()
    .rename("count")
    .reset_index()
)

label_distribution_path = PROCESSED_DIR / "label_distribution_by_split_config.csv"
breed_distribution_path = PROCESSED_DIR / "breed_distribution_by_split_config.csv"
label_distribution.to_csv(label_distribution_path, index=False)
breed_distribution.to_csv(breed_distribution_path, index=False)

label_distribution[label_distribution["split_config"] == PRIMARY_SPLIT_NAME]

In [ ]:
primary_label_distribution = label_distribution[
    label_distribution["split_config"] == PRIMARY_SPLIT_NAME
]
pivot_distribution = primary_label_distribution.pivot(index="split", columns="label", values="count")

ax = pivot_distribution.loc[["train", "validation", "test"]].plot(
    kind="bar",
    figsize=(8, 5),
    rot=0,
    color=["#4C78A8", "#F58518"],
)
ax.set_title("Dog/Cat Count by Split")
ax.set_xlabel("split")
ax.set_ylabel("count")
ax.legend(title="label")
plt.tight_layout()

figure_path = FIGURE_DIR / "class_distribution_split_70_15_15.png"
plt.savefig(figure_path, dpi=150)
plt.show()

print(figure_path)

## 11. 결과를 보고 판단하기

확인할 점:

1. train, validation, test가 의도한 비율에 가깝게 나뉘었는지 확인합니다.
2. 각 split 안에 dog/cat이 모두 들어 있는지 확인합니다.
3. 품종 기준 stratify 덕분에 각 품종이 split마다 들어갔는지 확인합니다.
4. 여러 split 비율 중 나중에 학습 결과 비교에 사용할 후보를 정리합니다.

이 조건이 만족되면 다음 학습 단계로 넘어갈 수 있습니다.

In [ ]:
split_summary = (
    all_splits_df.groupby(["split_config", "split"])
    .size()
    .rename("count")
    .reset_index()
)

split_summary["ratio"] = split_summary["count"] / split_summary.groupby("split_config")["count"].transform("sum")
split_summary

## 12. 발표에 넣을 정리 문장

이번 과제에서는 Kaggle처럼 이미 train/test가 나뉜 데이터셋 대신 Oxford-IIIT Pet Dataset을 사용했습니다. 그 이유는 데이터 분리 과정을 직접 경험하고, train/validation/test가 학습 과정에서 각각 어떤 역할을 하는지 설명하기 위해서입니다.

저는 전체 데이터를 dog/cat 이진 라벨로 변환한 뒤, 70/15/15 비율을 기본 split으로 사용했습니다. 또한 80/10/10과 60/20/20 split도 함께 만들어 나중에 학습 결과를 비교할 수 있게 했습니다.

이때 단순히 dog/cat 기준으로만 나누지 않고, 품종 기준 stratified split을 사용했습니다. 이렇게 하면 특정 품종이 train이나 test에 한쪽으로 몰리는 문제를 줄일 수 있습니다.

train set은 모델이 학습하는 데이터, validation set은 학습 중 모델 선택과 overfitting 여부를 확인하는 데이터, test set은 마지막 최종 성능 평가에 사용하는 데이터로 구분했습니다.

## 13. 이번 단계에서 배운 점

- 데이터셋을 고르는 기준도 과제 목적에 맞아야 합니다.
- train/validation/test는 단순히 비율로 나누는 것이 아니라 역할이 다릅니다.
- random seed는 같은 split을 재현하게 해주지만, 분포 균형을 직접 보장하지는 않습니다.
- 품종 기준 stratified split을 사용하면 dog/cat뿐 아니라 품종 쏠림도 줄일 수 있습니다.
- split을 여러 비율로 저장해두면 이후 학습 실험에서 같은 조건으로 비교할 수 있습니다.
- 클래스 분포를 확인해야 accuracy, precision, recall 해석이 더 신뢰할 수 있습니다.